---
## 1. Install & Import

In [ ]:
!pip install ultralytics albumentations timm gradio -q

In [ ]:
import os, json, cv2, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models

import albumentations as A
from albumentations.pytorch import ToTensorV2

from ultralytics import YOLO

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

---
## 2. Configuration & Paths

> Update `BEST_CLS_PATH` and `BEST_YOLO_PATH` to your Milestone 3 checkpoint locations.

In [ ]:
# -- Kaggle input paths (same structure as M2/M3) -------------------------
TRAIN_CSV  = "/kaggle/input/datasets/alaagastien/cliniscan2/train.csv"
META_CSV   = "/kaggle/input/datasets/alaagastien/cliniscan2/archive/train_meta.csv"
IMG_DIR    = "/kaggle/input/datasets/alaagastien/cliniscan2/archive/train"
TEST_DIR   = "/kaggle/input/cliniscan2/archive/test"
WORK_DIR   = "/kaggle/working"

# -- Checkpoints saved in Milestone 3 ------------------------------------
BEST_CLS_PATH  = os.path.join(WORK_DIR, 'cliniscan_best_cls_m3.pth')
BEST_YOLO_PATH = os.path.join(WORK_DIR, 'yolo_m3/weights/best.pt')
YAML_PATH      = "/kaggle/working/data_m3.yaml"

# -- Output directory for M4 artefacts -----------------------------------
M4_DIR = os.path.join(WORK_DIR, 'milestone4_outputs')
os.makedirs(M4_DIR, exist_ok=True)

# -- Class map (identical to M2/M3) --------------------------------------
CLASS_NAMES = {
    0:  'Aortic enlargement',  1: 'Atelectasis',       2: 'Calcification',
    3:  'Cardiomegaly',        4: 'Consolidation',     5: 'ILD',
    6:  'Infiltration',        7: 'Lung Opacity',      8: 'Nodule/Mass',
    9:  'Other lesion',       10: 'Pleural effusion', 11: 'Pleural thickening',
   12:  'Pneumothorax',       13: 'Pulmonary fibrosis'
}
NUM_CLASSES = 14
IMGSIZE     = 224
BATCH_SIZE  = 32

print('Config loaded.')
print(f'M4 output dir: {M4_DIR}')

---
## 3. Rebuild Val Split (same 80/20 stratified split as M3)

In [ ]:
df   = pd.read_csv(TRAIN_CSV)
meta = pd.read_csv(META_CSV)

df = df[df['class_id'] != 14]   # drop 'No finding'
df = df.dropna(subset=['x_min', 'y_min', 'x_max', 'y_max'])

# Single-label per image (same as M3)
df_single = df.sort_values('image_id').groupby('image_id').first().reset_index()

train_df, val_df = train_test_split(
    df_single, test_size=0.2,
    stratify=df_single['class_id'], random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)}')

In [ ]:
# Dataset class — identical to M3
class XrayDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None):
        self.df         = df
        self.img_dir    = img_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.img_dir, row['image_id'] + '.png')
        image = np.array(Image.open(path).convert('RGB'))
        label = int(row['class_id'])
        if self.transforms:
            image = self.transforms(image=image)['image']
        return image, label


val_transforms = A.Compose([
    A.Resize(IMGSIZE, IMGSIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std =(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_dataset = XrayDataset(val_df, IMG_DIR, val_transforms)
val_loader  = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=True)
print(f'Val loader ready — {len(val_dataset)} samples')

---
## 4. Rebuild & Load Best Classification Model

Exact same `build_model()` architecture from Milestone 3 (ResNet50 + custom head).

In [ ]:
def build_model(num_classes=NUM_CLASSES, dropout=0.4):
    """Identical to build_model() in Milestone 3."""
    model = models.resnet50(pretrained=False)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(dropout / 2),
        nn.Linear(256, num_classes)
    )
    return model


clf_model = build_model().to(device)
clf_model.load_state_dict(torch.load(BEST_CLS_PATH, map_location=device))
clf_model.eval()
print('Classification model loaded from:', BEST_CLS_PATH)

---
## 5. Final Classification Evaluation
### 5.1 Full Inference

In [ ]:
@torch.no_grad()
def evaluate_final(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in tqdm(loader, desc='Final inference'):
        images  = images.to(device)
        outputs = model(images)
        probs   = F.softmax(outputs, dim=1)
        preds   = torch.argmax(probs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


all_preds, all_labels, all_probs = evaluate_final(clf_model, val_loader)
print(f'Inference complete — {len(all_labels)} samples.')

### 5.2 Metrics Summary

In [ ]:
test_acc = accuracy_score(all_labels, all_preds)
test_f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
test_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')

print('=' * 55)
print('  CliniScan — Final Classification Results (M4)')
print('=' * 55)
print(f'  Accuracy      : {test_acc:.4f}')
print(f'  F1  (macro)   : {test_f1:.4f}')
print(f'  AUC (OvR)     : {test_auc:.4f}')
print('=' * 55)

print('\nPer-Class Report:')
print(classification_report(
    all_labels, all_preds,
    target_names=[CLASS_NAMES[i] for i in range(NUM_CLASSES)],
    zero_division=0
))

### 5.3 Per-Class AUC Table (saved to CSV)

In [ ]:
rows = []
for i in range(NUM_CLASSES):
    y_bin   = (all_labels == i).astype(int)
    y_score = all_probs[:, i]
    try:
        auc_i = roc_auc_score(y_bin, y_score)
    except ValueError:
        auc_i = float('nan')
    rows.append({
        'Class'  : CLASS_NAMES[i],
        'Support': int(y_bin.sum()),
        'F1'     : round(f1_score(all_labels, all_preds,
                                  labels=[i], average='macro',
                                  zero_division=0), 4),
        'AUC'    : round(auc_i, 4),
    })

metrics_df = pd.DataFrame(rows)
csv_path   = os.path.join(M4_DIR, 'classification_metrics.csv')
metrics_df.to_csv(csv_path, index=False)
print(metrics_df.to_string(index=False))
print(f'\nSaved to {csv_path}')

### 5.4 ROC Curves — All 14 Classes

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(22, 13))
axes = axes.flatten()

for i in range(NUM_CLASSES):
    ax      = axes[i]
    y_bin   = (all_labels == i).astype(int)
    y_score = all_probs[:, i]
    try:
        fpr, tpr, _ = roc_curve(y_bin, y_score)
        auc_val     = roc_auc_score(y_bin, y_score)
        ax.plot(fpr, tpr, color='steelblue', lw=2,
                label=f'AUC = {auc_val:.3f}')
        ax.plot([0, 1], [0, 1], 'k--', lw=1)
        ax.legend(loc='lower right', fontsize=8)
    except ValueError:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center')
    ax.set_title(CLASS_NAMES[i], fontsize=9)
    ax.set_xlabel('FPR', fontsize=7)
    ax.set_ylabel('TPR', fontsize=7)

axes[-1].set_visible(False)
plt.suptitle('CliniScan — Per-Class ROC Curves (Milestone 4)', fontsize=14, y=1.01)
plt.tight_layout()
roc_path = os.path.join(M4_DIR, 'roc_curves.png')
plt.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {roc_path}')

### 5.5 Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=[CLASS_NAMES[i] for i in range(NUM_CLASSES)],
    yticklabels=[CLASS_NAMES[i] for i in range(NUM_CLASSES)]
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual',    fontsize=11)
ax.set_title('Confusion Matrix — CliniScan Final Evaluation (M4)', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0,  fontsize=8)
plt.tight_layout()
cm_path = os.path.join(M4_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {cm_path}')

---
## 6. Grad-CAM Heatmap Gallery

Uses the exact same custom `GradCAM` class from Milestone 3 (hooked on `layer4[-1]`).

In [ ]:
class GradCAM:
    """Gradient-weighted Class Activation Mapping — same implementation as M3."""

    def __init__(self, model, target_layer):
        self.model       = model
        self.gradients   = None
        self.activations = None
        target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o.detach()))
        target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(1).item()
        self.model.zero_grad()
        output[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam     = F.relu((weights * self.activations).sum(1, keepdim=True))
        cam     = F.interpolate(cam, size=input_tensor.shape[2:],
                                mode='bilinear', align_corners=False)
        cam     = cam.squeeze().cpu().numpy()
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam, class_idx


grad_cam = GradCAM(clf_model, clf_model.layer4[-1])
print('GradCAM hooked on clf_model.layer4[-1]')

In [ ]:
def denormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = tensor.cpu().numpy().transpose(1, 2, 0) * std + mean
    return np.clip(img, 0, 1)


NUM_GRADCAM = 8
sample_idx  = np.random.choice(len(val_dataset), NUM_GRADCAM, replace=False)

fig, axes = plt.subplots(NUM_GRADCAM, 3, figsize=(14, 4.5 * NUM_GRADCAM))
fig.suptitle('Grad-CAM Heatmaps — Final Evaluation Samples (M4)',
             fontsize=13, y=1.005)

for row, idx in enumerate(tqdm(sample_idx, desc='Grad-CAM')):
    img_t, true_label = val_dataset[idx]
    inp = img_t.unsqueeze(0).to(device)

    cam, pred_label = grad_cam.generate(inp)

    orig  = denormalize(img_t)
    hm    = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    hm    = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB) / 255.0
    blend = np.clip(0.55 * orig + 0.45 * hm, 0, 1)

    correct_mark = 'CORRECT' if pred_label == true_label else 'WRONG'

    axes[row, 0].imshow(orig)
    axes[row, 0].set_title(f'True: {CLASS_NAMES[true_label]}', fontsize=9)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(orig)
    axes[row, 1].imshow(cam, cmap='jet', alpha=0.5)
    axes[row, 1].set_title('Grad-CAM Overlay', fontsize=9)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(blend)
    axes[row, 2].set_title(
        f'[{correct_mark}] Pred: {CLASS_NAMES[pred_label]}', fontsize=9)
    axes[row, 2].axis('off')

plt.tight_layout()
cam_path = os.path.join(M4_DIR, 'gradcam_gallery_m4.png')
plt.savefig(cam_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {cam_path}')

---
## 7. YOLOv11s — Final Detection Evaluation

In [ ]:
yolo_best = YOLO(BEST_YOLO_PATH)

# Run val on the same YAML as M3
metrics = yolo_best.val(
    data     = YAML_PATH,
    imgsz    = 640,
    conf     = 0.25,
    iou      = 0.5,
    device   = 0 if torch.cuda.is_available() else 'cpu',
    project  = M4_DIR,
    name     = 'yolo_final_eval',
    exist_ok = True,
    save_json= True,
)

print()
print('=' * 50)
print('  CliniScan — Final Detection Results (M4)')
print('=' * 50)
print(f'  mAP@0.50      : {metrics.box.map50:.4f}')
print(f'  mAP@0.50:0.95 : {metrics.box.map:.4f}')
print(f'  Precision     : {metrics.box.mp:.4f}')
print(f'  Recall        : {metrics.box.mr:.4f}')
print('=' * 50)

### 7.1 GT vs Prediction Visualisation (val samples)

Reuses the exact `draw_boxes` helper from Milestone 3.

In [ ]:
COLORS = plt.cm.get_cmap('tab20', NUM_CLASSES)

def draw_boxes(ax, boxes, labels, title, scores=None):
    for i, (box, lbl) in enumerate(zip(boxes, labels)):
        x1, y1, x2, y2 = box
        color = COLORS(int(lbl))[:3]
        rect  = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        text = CLASS_NAMES[int(lbl)]
        if scores is not None:
            text += f' {scores[i]:.2f}'
        ax.text(x1, y1 - 4, text, color=color, fontsize=7,
                bbox=dict(facecolor='black', alpha=0.35, pad=1))
    ax.set_title(title, fontsize=9)
    ax.axis('off')


df_bbox  = pd.read_csv(TRAIN_CSV)
df_bbox  = df_bbox[df_bbox['class_id'] != 14]
df_bbox  = df_bbox.dropna(subset=['x_min', 'y_min', 'x_max', 'y_max'])
val_ids  = list(val_df['image_id'].unique())
sample_ids = val_ids[:6]

fig, axes = plt.subplots(len(sample_ids), 2,
                         figsize=(14, 5.5 * len(sample_ids)))
fig.suptitle('Detection: Ground Truth vs Predictions — M4 Final',
             fontsize=13, y=1.01)

for row_i, img_id in enumerate(sample_ids):
    img_path = os.path.join(IMG_DIR, img_id + '.png')
    img      = np.array(Image.open(img_path).convert('RGB'))

    gt       = df_bbox[df_bbox['image_id'] == img_id]
    gt_boxes = gt[['x_min', 'y_min', 'x_max', 'y_max']].values.tolist()
    gt_lbls  = gt['class_id'].values.tolist()

    result   = yolo_best.predict(img_path, conf=0.25, verbose=False)[0]
    pd_boxes = result.boxes.xyxy.cpu().numpy().tolist()
    pd_lbls  = result.boxes.cls.cpu().numpy().tolist()
    pd_conf  = result.boxes.conf.cpu().numpy().tolist()

    axes[row_i, 0].imshow(img)
    draw_boxes(axes[row_i, 0], gt_boxes, gt_lbls, f'GT  {img_id[:18]}')
    axes[row_i, 1].imshow(img)
    draw_boxes(axes[row_i, 1], pd_boxes, pd_lbls, 'Predicted', scores=pd_conf)

plt.tight_layout()
det_path = os.path.join(M4_DIR, 'detection_gt_vs_pred_m4.png')
plt.savefig(det_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {det_path}')

---
## 8. Model Export
### 8.1 ResNet50 Classifier → ONNX

In [ ]:
import onnx
import onnxruntime as ort

clf_model.eval().cpu()
dummy     = torch.randn(1, 3, IMGSIZE, IMGSIZE)
onnx_path = os.path.join(M4_DIR, 'cliniscan_resnet50.onnx')

torch.onnx.export(
    clf_model, dummy, onnx_path,
    opset_version = 17,
    input_names   = ['input'],
    output_names  = ['logits'],
    dynamic_axes  = {'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    export_params = True,
)

onnx.checker.check_model(onnx.load(onnx_path))

sess    = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
ort_out = sess.run(None, {'input': dummy.numpy()})
print(f'ONNX output shape : {ort_out[0].shape}')   # (1, 14)
print(f'Saved to {onnx_path}')

clf_model = clf_model.to(device)

### 8.2 ResNet50 Classifier → TorchScript

In [ ]:
clf_model.eval().cpu()
ts_path  = os.path.join(M4_DIR, 'cliniscan_resnet50.torchscript.pt')
scripted = torch.jit.trace(clf_model, dummy)
torch.jit.save(scripted, ts_path)

loaded = torch.jit.load(ts_path)
out    = loaded(dummy)
print(f'TorchScript output shape : {out.shape}')   # torch.Size([1, 14])
print(f'Saved to {ts_path}')

clf_model = clf_model.to(device)

### 8.3 YOLOv11s → ONNX

In [ ]:
yolo_onnx_path = yolo_best.export(format='onnx', imgsz=640, opset=17)
print(f'YOLOv11s ONNX saved to {yolo_onnx_path}')

---
# 9. Gradio Deployment Demo





In [ ]:
import gradio as gr

# Inference transform — no augmentation
infer_transform = A.Compose([
    A.Resize(IMGSIZE, IMGSIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std =(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

clf_model.eval().to(device)
grad_cam_serve = GradCAM(clf_model, clf_model.layer4[-1])


def cliniscan_predict(pil_image):
    """Full CliniScan inference pipeline for one uploaded X-ray."""

    # 1. Pre-process
    img_np = np.array(pil_image.convert('RGB'))
    tensor = infer_transform(image=img_np)['image']
     inp    = tensor.unsqueeze(0).to(device)

    # 2. Classification
    with torch.no_grad():
        probs = F.softmax(clf_model(inp), dim=1).squeeze().cpu().numpy()
    sorted_idx = np.argsort(probs)[::-1]
    label_dict = {CLASS_NAMES[i]: float(round(probs[i], 4)) for i in sorted_idx}

    # 3. Grad-CAM
    cam, top_cls = grad_cam_serve.generate(inp, class_idx=int(sorted_idx[0]))
    orig_r       = cv2.resize(img_np, (IMGSIZE, IMGSIZE)).astype(np.float32) / 255.0
    hm           = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    hm           = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB) / 255.0
    blend        = np.clip(0.55 * orig_r + 0.45 * hm, 0, 1)
    cam_pil      = Image.fromarray((blend * 255).astype(np.uint8))

    # 4. YOLO detection
    tmp_path = '/tmp/cliniscan_upload.png'
    pil_image.save(tmp_path)
    det_result = yolo_best.predict(tmp_path, conf=0.25, verbose=False)[0]
    det_img    = img_np.copy()
    for box in det_result.boxes:
       x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].cpu().numpy()]
        cls_idx         = int(box.cls.item())
        conf            = float(box.conf.item())
        color           = (57, 255, 20)
        cv2.rectangle(det_img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            det_img, f'{CLASS_NAMES[cls_idx]} {conf:.2f}',
            (x1, max(y1 - 5, 12)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1
        )
    det_pil = Image.fromarray(det_img)

    return label_dict, cam_pil, det_pil


# Build Gradio UI
with gr.Blocks(title='CliniScan') as demo:
    gr.Markdown("""
    # CliniScan - Lung Abnormality Detection
    ### Milestone 4 Deployment Demo
    Upload a chest X-ray to get classification probabilities,
    a Grad-CAM heatmap, and YOLOv11s detection boxes.
    """)
 with gr.Row():
        out_label = gr.Label(num_top_classes=5, label='Top-5 Predicted Findings')

    with gr.Row():
        out_cam = gr.Image(label='Grad-CAM Heatmap (Top Prediction)')
        out_det = gr.Image(label='YOLOv11s Detection Output')

    gr.Markdown("""
    ---
    Disclaimer: This tool is for research / educational purposes only.
    It is NOT a substitute for professional radiological assessment.
    """)

    run_btn.click(
        fn      = cliniscan_predict,
        inputs  = [inp_img],
        outputs = [out_label, out_cam, out_det]
    )

demo.launch(share=True)